# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [ ]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    "table": ["employees", "salary", "studies"],
    "columns": [
        "employee_id, first_name, last_name, gender, date_of_birth, email, phone_number, address, city, state, country, department, job_title, manager_id, hire_date, employment_status",
        "salary_id, employee_id, year, base_salary, bonus, allowance, overtime_pay, total_salary",
        "study_id, employee_id, institution_name, degree, field_of_study, level_of_study, type_of_study, start_date, end_date, graduation_year, grade, country"
    ],
    "definition": [
        "Employee information including personal details, contact information, department, job title, manager, hire date, and employment status.",
        "Salary details for each employee by year, including base salary, bonus, allowances, overtime pay, and total salary.",
        "Educational background of employees, including institution name, degree, field of study, study level, type of study, study duration, graduation year, grade, and country."
    ]
}




df = pd.DataFrame(data)
print(df)

       table                                            columns  \
0  employees  employee_id, first_name, last_name, gender, da...   
1     salary  salary_id, employee_id, year, base_salary, bon...   
2    studies  study_id, employee_id, institution_name, degre...   

                                          definition  
0  Employee information including personal detail...  
1  Salary details for each employee by year, incl...  
2  Educational background of employees, including...  


In [6]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [7]:
print(text_tables)

employees: Employee information including personal details, contact information, department, job title, manager, hire date, and employment status.
salary: Salary details for each employee by year, including base salary, bonus, allowances, overtime pay, and total salary.
studies: Educational background of employees, including institution name, degree, field of study, study level, type of study, study duration, graduation year, grade, and country.


In [8]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [10]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="What is the average salary of employees by department?")

In [11]:
print(return_OAI(pqt1))

{
    "tables": ["employees", "salary"]
}


In [12]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="List all employees and their departments.")

In [13]:
print(return_OAI(pqt3))

{
  "tables": ["employees"]
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [14]:
prompt_question_tables = """
You are an AI assistant.

The available database tables are:

{tables}

Question:
{question}

Return the names of the tables that are needed to answer the question.
"""

In [16]:
question = "Which employees have a Master's degree?"
print(return_OAI(prompt_question_tables.format(tables=text_tables, question=question)))

To determine which employees have a Master's degree, we would need the "employees" table and the "studies" table. The "studies" table contains information about the educational background of employees, including their degree details such as study level (which includes Master's degree information). The "employees" table is needed to link the employee information with their educational background from the "studies" table.


In [17]:
question = "Which employee has the highest salary?"
print(return_OAI(prompt_question_tables.format(tables=text_tables, question=question)))

To determine which employee has the highest salary, we would need the "employees" table and the "salary" table. These tables contain the necessary information about employees and their corresponding salaries.


In [18]:
prompt_question_tables = """
You are a database expert.

Below are the available database tables and their descriptions.

{tables}

Instructions:
- Identify only the tables required to answer the user's question.
- Do not invent tables.
- If no table can answer the question, reply:
'I do not know.'

Question:
{question}

Relevant Tables:
"""

In [19]:
question = "Show each employee's salary and degree."
print(return_OAI(prompt_question_tables.format(tables=text_tables, question=question)))

employees, salary, studies


In [20]:
Question="Which university did John attend?"
print(return_OAI(prompt_question_tables.format(tables=text_tables, question=Question)))

studies
